# nanochat base train + eval notebook

Notebook-first workflow for `scripts/base_train.py` **and** `scripts/base_eval.py` without manual CLI typing.

## 1) Configure training
Set token budget + evaluation cadence. CORE eval is enabled by default via `core_metric_every`.


In [ ]:
from math import ceil

TRAINING_CONFIG = {
    # Logging/runtime
    'run': 'nb-base-train',
    'data_dir': None,
    'checkpoints_dir': None,
    'device_type': '',

    # Model
    'depth': 20,
    'aspect_ratio': 64,
    'head_dim': 128,
    'max_seq_len': 2048,
    'window_pattern': 'SSSL',

    # Batch + horizon
    'device_batch_size': 8,
    'total_batch_size': 524288,
    'training_tokens': 20 * 524288,  # primary budget knob

    # Optimization
    'embedding_lr': 0.3,
    'unembedding_lr': 0.004,
    'weight_decay': 0.2,
    'matrix_lr': 0.02,
    'scalar_lr': 0.5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.95,
    'warmup_ratio': 0.0,
    'warmdown_ratio': 0.5,
    'final_lr_frac': 0.0,

    # In-training evaluation (base_train internals)
    'eval_every': 250,
    'eval_tokens': 2 * 524288,
    'core_metric_every': 2000,            # set -1 to disable CORE inside training
    'core_metric_max_per_task': 500,
    'sample_every': 2000,
    'save_every': 1000,

    # Extras
    'resume_from_step': -1,
    'model_tag': 'nb_d20',
    'fp8': False,
    'fp8_recipe': 'tensorwise',
}

TRAINING_CONFIG['num_iterations'] = ceil(TRAINING_CONFIG['training_tokens'] / TRAINING_CONFIG['total_batch_size'])
print('Planned iterations:', TRAINING_CONFIG['num_iterations'])
print('Planned tokens   :', TRAINING_CONFIG['num_iterations'] * TRAINING_CONFIG['total_batch_size'])
print('CORE in training?:', TRAINING_CONFIG['core_metric_every'])

## 2) Convert config to argv for base_train


In [ ]:
def base_train_argv(cfg: dict) -> list[str]:
    flag_map = {
        'run': '--run', 'device_type': '--device-type',
        'data_dir': '--data-dir', 'checkpoints_dir': '--checkpoints-dir',
        'depth': '--depth', 'aspect_ratio': '--aspect-ratio', 'head_dim': '--head-dim',
        'max_seq_len': '--max-seq-len', 'window_pattern': '--window-pattern',
        'num_iterations': '--num-iterations',
        'device_batch_size': '--device-batch-size', 'total_batch_size': '--total-batch-size',
        'embedding_lr': '--embedding-lr', 'unembedding_lr': '--unembedding-lr',
        'weight_decay': '--weight-decay', 'matrix_lr': '--matrix-lr', 'scalar_lr': '--scalar-lr',
        'adam_beta1': '--adam-beta1', 'adam_beta2': '--adam-beta2',
        'warmup_ratio': '--warmup-ratio', 'warmdown_ratio': '--warmdown-ratio', 'final_lr_frac': '--final-lr-frac',
        'resume_from_step': '--resume-from-step',
        'eval_every': '--eval-every', 'eval_tokens': '--eval-tokens',
        'core_metric_every': '--core-metric-every', 'core_metric_max_per_task': '--core-metric-max-per-task',
        'sample_every': '--sample-every', 'save_every': '--save-every',
        'model_tag': '--model-tag', 'fp8_recipe': '--fp8-recipe',
    }
    argv = ['scripts.base_train']
    if cfg.get('fp8', False):
        argv.append('--fp8')
    for k, flag in flag_map.items():
        if k in cfg:
            argv.extend([flag, str(cfg[k])])
    return argv

train_argv = base_train_argv(TRAINING_CONFIG)
print(' '.join(train_argv[:14]), '...')

## 3) Run base_train (includes bpb eval + CORE eval per cadence)
`core_metric_every` and `eval_every` control in-training evaluation.


In [ ]:
import runpy, sys
old_argv = sys.argv[:]
sys.argv = train_argv
try:
    runpy.run_module('scripts.base_train', run_name='__main__')
finally:
    sys.argv = old_argv

## 4) Optional: run standalone base_eval.py after training
Use this when you want a full explicit eval pass after training completes.


In [ ]:
BASE_EVAL_CONFIG = {
    'device_type': TRAINING_CONFIG['device_type'],
    'model_tag': TRAINING_CONFIG['model_tag'],
    'step': None,  # None = latest checkpoint
    'eval': 'core,bpb,sample',
    'max_per_task': 500,
    'device_batch_size': 16,
    'split_tokens': 8 * 524288,
}

def base_eval_argv(cfg: dict) -> list[str]:
    flag_map = {
        'device_type': '--device-type',
        'model_tag': '--model-tag',
        'step': '--step',
        'eval': '--eval',
        'max_per_task': '--max-per-task',
        'device_batch_size': '--device-batch-size',
        'split_tokens': '--split-tokens',
    }
    argv = ['scripts.base_eval']
    for k, flag in flag_map.items():
        if k not in cfg or cfg[k] is None:
            continue
        argv.extend([flag, str(cfg[k])])
    return argv

eval_argv = base_eval_argv(BASE_EVAL_CONFIG)
print(' '.join(eval_argv))

In [ ]:
# Uncomment to run explicit post-training eval:
# import runpy, sys
# old_argv = sys.argv[:]
# sys.argv = eval_argv
# try:
#     runpy.run_module('scripts.base_eval', run_name='__main__')
# finally:
#     sys.argv = old_argv

## Notes
- If you set `core_metric_every=-1`, training will skip CORE eval (this was the main reason it can appear missing).
- For quick smoke tests, lower depth/sequence/batch/token budget dramatically.
- Multi-GPU distributed runs are still best launched from terminal with `torchrun`.
